In [ ]:
from pyspark.sql import DataFrame, functions as F

from transforms.dimensions import (
    LEAGUE_COUNTRY,
    build_dim_team,
    build_dim_date,
    build_dim_league,
    build_dim_season,
)

SILVER_TABLE = "stryktipset.silver.matches"
DIM_TEAM_TABLE = "stryktipset.gold.dim_team"
DIM_DATE_TABLE = "stryktipset.gold.dim_date"
DIM_LEAGUE_TABLE = "stryktipset.gold.dim_league"
DIM_SEASON_TABLE = "stryktipset.gold.dim_season"


In [ ]:
def write_dimension(df: DataFrame, table: str) -> None:
    """Full overwrite, same as 02_silver_transform does with matches --
    no incremental merge needed."""
    df.write.format('delta').mode('overwrite').saveAsTable(table)

In [ ]:
def main():
    df = spark.table(SILVER_TABLE)
    
    league_lookup = spark.createDataFrame(list(LEAGUE_COUNTRY.items()), ['league', 'country'])
    df_with_country = df.join(league_lookup, on='league', how='left')

    write_dimension(build_dim_team(df), DIM_TEAM_TABLE)
    write_dimension(build_dim_date(df), DIM_DATE_TABLE)
    write_dimension(build_dim_league(spark), DIM_LEAGUE_TABLE)
    write_dimension(
        build_dim_season(df_with_country.filter(F.col('country').isNotNull())),
        DIM_SEASON_TABLE,
    )

main()